# The invariant harmonic sector is not always zero

### $\epsilon_{\mathrm{harm}}\neq0$, a counting obstruction that forces it, and the predicate-dependence of the vanishing phase

**Dean A. Kulik**, QuHarmonics Research Group — ORCID 0009-0003-3128-8828
Worked with Claude (Opus 5). September 2026. CC BY-NC 4.0.

---

## Abstract

*Transformation without loss* §2 records, as measured-not-proved, that the invariant harmonic space
vanishes: $\epsilon_{\mathrm{harm}}=0$ across three arenas and both standard 2-cell predicates, so
"the entire invariant non-exact sector is coexact." §12 asks for $\epsilon_{\mathrm{harm}}\neq0$ or a
proof it cannot occur, calling it *the single result that would reopen a non-coexact channel*, with
high-girth, small-automorphism substrates named as candidates.

This notebook reopens the channel and then partly closes the resulting question.

1. **$\epsilon_{\mathrm{harm}}\neq0$ occurs.** Minimal witness the **bull** — triangle with two
   pendants, $|V|=5$, $|\mathrm{Aut}|=2$, $\epsilon=2$, $\epsilon_{\mathrm{harm}}=1$ — exhaustively
   minimal over all connected graphs on $\le5$ vertices.
2. **The candidate class was wrong.** Girth is irrelevant: Petersen (5), $C_8$+tail (8), $C_9$ (9) all
   vanish; the witness has girth 3. **Trees** give $\epsilon_{\mathrm{harm}}>0$, so the surviving
   sector is not substrate winding — it is multi-token configuration structure.
3. **A counting obstruction forces existence.** Since $d_1d_0=0$,
   $$\epsilon_{\mathrm{harm}}=\epsilon-\operatorname{rank}(d_1|_{C^1_G})\ \ge\ \epsilon-\dim C^2_G,$$
   so $\epsilon>\dim C^2_G\Rightarrow\epsilon_{\mathrm{harm}}>0$. Elementary, and vacuous on every
   previously tested arena, which is why it was not written. It certifies **85 of 98** observed
   nonzero cases (87%) with no rank computation and no false positives.
4. **The vanishing side is predicate-dependent.** $\#(V/\mathrm{Aut})\le2\Rightarrow
   \epsilon_{\mathrm{harm}}=0$ holds $112/112$ for the all-4-cycle complex and is **false** for the
   disjoint-support-square complex, with a minimal counterexample at $|V|=8$. The conjecture's truth
   value is a property of the 2-cell predicate, not of $X$.

Consequently $(X,K,G,\text{2-cell predicate})$ is the minimal data record; a quoted
$\epsilon_{\mathrm{harm}}$ without the predicate is incomplete.

---

## Status

**Proved** — the counting obstruction; the identity $\epsilon_{\mathrm{harm}}=\epsilon-\operatorname{rank}(d_1|_{C^1_G})$;
per-case vanishing certificates by exact rank (see §6 for why mod-$p$ suffices).

**Measured** — the witnesses, the exhaustive minimality of the bull, the phase table, the level profiles.

**Conjectural** — $\le2$ vertex orbits $\Rightarrow\epsilon_{\mathrm{harm}}=0$, **for the all-4-cycle
predicate only**.

**Killed during this work, both mine** — see §8. Neither reached a claim.

**Not established** — nothing physical. $\epsilon_{\mathrm{harm}}$ is a dimension.
*Transformation without loss* §10 stands unchanged.

## 1. Library

Self-contained: `itertools`, `collections`, `numpy`, `fractions`. Automorphism groups, arenas,
2-cell complexes under both predicates, the invariant Hodge decomposition, the spectator-level
filtration, and exact/modular rank are all built here from scratch.

In [1]:
"""
Capacity-defect engine for token-configuration arenas.
Self-contained: itertools, collections, numpy only.

Implements, from scratch:
  - automorphism groups of finite simple graphs (backtracking)
  - the capacity arenas X^(r) (distinguished site m holds up to r tokens)
  - the invariant residual  eps = dim R_G  (orbit count AND independent rank computation)
  - the capacity defect D_r = ker(sigma_r)
  - the cone-quotient graph Q_r and beta_1(Q_r)
"""
import itertools
from collections import defaultdict, deque
import numpy as np

TOL = 1e-9

# ---------------------------------------------------------------- graphs
def mkgraph(n, edges):
    adj = {v: set() for v in range(n)}
    for a, b in edges:
        adj[a].add(b); adj[b].add(a)
    return adj

def cycle(n):   return mkgraph(n, [(i, (i+1) % n) for i in range(n)])
def path(n):    return mkgraph(n, [(i, i+1) for i in range(n-1)])
def complete(n):return mkgraph(n, [(i, j) for i in range(n) for j in range(i+1, n)])
def star(n):    return mkgraph(n, [(0, i) for i in range(1, n)])

def qcube3():
    E = []
    for a in range(8):
        for b in range(a+1, 8):
            if bin(a ^ b).count('1') == 1: E.append((a, b))
    return mkgraph(8, E)

def prism():  # C3 x K2
    return mkgraph(6, [(0,1),(1,2),(2,0),(3,4),(4,5),(5,3),(0,3),(1,4),(2,5)])

def petersen():
    out = [(i, (i+1) % 5) for i in range(5)]
    inn = [(5+i, 5+((i+2) % 5)) for i in range(5)]
    sp  = [(i, 5+i) for i in range(5)]
    return mkgraph(10, out+inn+sp)

def paw():    return mkgraph(4, [(0,1),(1,2),(2,0),(0,3)])   # triangle + pendant at 0
def tritail():return mkgraph(5, [(0,1),(1,2),(2,0),(0,3),(3,4)])
def bowtie(): return mkgraph(5, [(0,1),(1,2),(2,0),(0,3),(3,4),(4,0)])
def theta():  return mkgraph(5, [(0,1),(1,2),(2,3),(3,4),(4,0),(0,2)])
def kite():   return mkgraph(4, [(0,1),(1,2),(2,0),(0,3),(1,3)])  # K4 minus an edge
def cbar(a,b):  # two cycles joined by a bridge: C_a --edge-- C_b
    E = [(i,(i+1)%a) for i in range(a)]
    E += [(a+i, a+((i+1)%b)) for i in range(b)]
    E += [(0, a)]
    return mkgraph(a+b, E)
def wedge(a,b):  # two cycles glued at a vertex (vertex 0)
    E = [(i,(i+1)%a) for i in range(a)]
    V = a
    ring = [0] + list(range(V, V+b-1))
    E += [(ring[i], ring[(i+1)%b]) for i in range(b)]
    return mkgraph(a+b-1, E)

def induced_minus(adj, m):
    vs = [v for v in adj if v != m]
    idx = {v: i for i, v in enumerate(vs)}
    E = []
    for v in vs:
        for w in adj[v]:
            if w != m and idx[v] < idx[w]: E.append((idx[v], idx[w]))
    return mkgraph(len(vs), E), idx, vs

def components(adj):
    seen, comps = set(), []
    for s in adj:
        if s in seen: continue
        c, q = set(), deque([s]); seen.add(s)
        while q:
            x = q.popleft(); c.add(x)
            for y in adj[x]:
                if y not in seen: seen.add(y); q.append(y)
        comps.append(c)
    return comps

# ------------------------------------------------------- automorphisms
def automorphisms(adj, fix=None):
    """All automorphisms of adj (as tuples perm[v]=image). fix: vertex that must be fixed."""
    n = len(adj)
    deg = {v: len(adj[v]) for v in adj}
    order = sorted(adj, key=lambda v: (-deg[v], v))
    out = []
    perm = {}
    used = set()

    def bt(i):
        if i == len(order):
            out.append(tuple(perm[v] for v in range(n))); return
        v = order[i]
        cands = [fix] if (fix is not None and v == fix) else [w for w in adj if deg[w] == deg[v] and w not in used]
        if fix is not None and v != fix: cands = [w for w in cands if w != fix]
        for w in cands:
            if fix is not None and v == fix and w != fix: continue
            ok = True
            for u in order[:i]:
                if (u in adj[v]) != (perm[u] in adj[w]): ok = False; break
            if ok:
                perm[v] = w; used.add(w)
                bt(i+1)
                del perm[v]; used.discard(w)
    bt(0)
    return out

def stab(adj, m):
    return automorphisms(adj, fix=m)

# ----------------------------------------------------------- the arena
# configuration = (j, U)   j = occupancy at m,  U = frozenset of occupied non-m sites
def arena(adj, m, K, r):
    """Arena X^(r): m holds up to r tokens, others at most 1, K tokens total."""
    others = [v for v in adj if v != m]
    V = []
    for j in range(0, r+1):
        k = K - j
        if 0 <= k <= len(others):
            for U in itertools.combinations(others, k):
                V.append((j, frozenset(U)))
    Vset = set(V)
    E = set()
    for (j, U) in V:
        for u in U:                                   # non-m -> non-m
            for w in adj[u]:
                if w != m and w not in U:
                    t = (j, frozenset(U - {u} | {w}))
                    if t in Vset: E.add(frozenset({(j, U), t}))
        if j >= 1:                                    # m -> neighbour
            for w in adj[m]:
                if w not in U:
                    t = (j-1, frozenset(U | {w}))
                    if t in Vset: E.add(frozenset({(j, U), t}))
        for u in (U & adj[m]):                        # neighbour -> m
            t = (j+1, frozenset(U - {u}))
            if t in Vset: E.add(frozenset({(j, U), t}))
    return V, [tuple(sorted(e, key=lambda c: (c[0], sorted(c[1])))) for e in E]

def act(g, cfg):
    j, U = cfg
    return (j, frozenset(g[u] for u in U))

def orbits_of(items, G, action):
    seen, reps, lab = set(), [], {}
    for x in items:
        if x in seen: continue
        orb = set()
        for g in G:
            orb.add(action(g, x))
        for y in orb: seen.add(y); lab[y] = len(reps)
        reps.append((x, orb))
    return reps, lab

def edge_action(g, e):
    a, b = e
    a2, b2 = act(g, a), act(g, b)
    return tuple(sorted((a2, b2), key=lambda c: (c[0], sorted(c[1]))))

def residual_orbitcount(V, E, G):
    """eps = (#non-reversible edge orbits) - (#vertex orbits) + (#components of quotient)."""
    vreps, vlab = orbits_of(V, G, act)
    ereps, elab = orbits_of(E, G, edge_action)
    nonrev = []
    for (e, orb) in ereps:
        a, b = e
        rev = any(act(g, a) == b and act(g, b) == a for g in G)
        if not rev: nonrev.append(e)
    # components of quotient graph (self-loops from reversible orbits do not matter)
    par = list(range(len(vreps)))
    def find(x):
        while par[x] != x: par[x] = par[par[x]]; x = par[x]
        return x
    for (e, orb) in ereps:
        a, b = e
        ra, rb = find(vlab[a]), find(vlab[b])
        if ra != rb: par[ra] = rb
    c = len({find(i) for i in range(len(vreps))})
    return len(nonrev) - len(vreps) + c, dict(E_nr=len(nonrev), V_orb=len(vreps), c=c,
                                              E_orb=len(ereps), vlab=vlab, elab=elab,
                                              ereps=ereps, vreps=vreps, nonrev=nonrev)

def residual_rank(V, E, G):
    """Independent measurement: dim C^1_G - rank(d_0 restricted to invariants)."""
    vreps, vlab = orbits_of(V, G, act)
    ereps, elab = orbits_of(E, G, edge_action)
    # invariant 1-cochain basis: one coordinate per non-reversible edge orbit
    keep = []
    for i, (e, orb) in enumerate(ereps):
        a, b = e
        if not any(act(g, a) == b and act(g, b) == a for g in G): keep.append(i)
    kidx = {i: p for p, i in enumerate(keep)}
    # d_0 on invariant 0-cochains (functions on vertex orbits)
    M = np.zeros((len(keep), len(vreps)))
    for i in keep:
        e, _ = ereps[i]
        a, b = e
        # orient a->b ; value = f(b)-f(a)
        M[kidx[i], vlab[b]] += 1.0
        M[kidx[i], vlab[a]] -= 1.0
    rank = 0 if M.size == 0 else int(np.linalg.matrix_rank(M, tol=TOL))
    return len(keep) - rank, len(keep), rank

# --------------------------------------------------- cone quotient Q_r
def cone_quotient(adj, m, K, r, G):
    """Build Q_r: orbits of the new layer + cone point; internal + cross edge orbits,
       reversible orbits deleted.  Returns (E_Q, V_Q, c_Q, beta1)."""
    others = [v for v in adj if v != m]
    k = K - r - 1
    if k < 0 or k > len(others): return None
    layer = [(r+1, frozenset(U)) for U in itertools.combinations(others, k)]
    lset = set(layer)
    vreps, vlab = orbits_of(layer, G, act)
    internal, cross = set(), set()
    for (j, U) in layer:
        for u in U:
            for w in adj[u]:
                if w != m and w not in U:
                    t = (j, frozenset(U - {u} | {w}))
                    if t in lset: internal.add(frozenset({(j, U), t}))
        for w in adj[m]:
            if w not in U:
                cross.add(((j, U), w))      # move m -> w, lands in X^(r) == cone point
    internal = [tuple(sorted(e, key=lambda c: (c[0], sorted(c[1])))) for e in internal]
    ireps, ilab = orbits_of(internal, G, edge_action)
    nr_int = []
    for (e, orb) in ireps:
        a, b = e
        if not any(act(g, a) == b and act(g, b) == a for g in G): nr_int.append(e)
    # cross edge orbits: (config, neighbour) pairs; never reversible (endpoints differ in j)
    creps, clab = orbits_of(list(cross), G, lambda g, x: (act(g, x[0]), g[x[1]]))
    V_Q = len(vreps) + 1
    E_Q = len(nr_int) + len(creps)
    # components
    par = list(range(V_Q)); CONE = len(vreps)
    def find(x):
        while par[x] != x: par[x] = par[par[x]]; x = par[x]
        return x
    def uni(x, y):
        rx, ry = find(x), find(y)
        if rx != ry: par[rx] = ry
    for (e, orb) in ireps:
        a, b = e; uni(vlab[a], vlab[b])
    for (x, orb) in creps:
        uni(vlab[x[0]], CONE)
    c_Q = len({find(i) for i in range(V_Q)})
    return dict(E_Q=E_Q, V_Q=V_Q, c_Q=c_Q, beta1=E_Q - V_Q + c_Q,
                n_int_orb=len(ireps), n_int_nr=len(nr_int), n_cross=len(creps),
                n_layer_orb=len(vreps))

# ------------------------------------------------- restriction sigma_r
def capacity_transition(adj, m, K, r, G=None):
    if G is None: G = stab(adj, m)
    Vu, Eu = arena(adj, m, K, r+1)
    Vd, Ed = arena(adj, m, K, r)
    if len(Vd) == 0: return None
    eu, iu = residual_orbitcount(Vu, Eu, G)
    ed, idd = residual_orbitcount(Vd, Ed, G)
    eu2, dimC1u, rku = residual_rank(Vu, Eu, G)
    ed2, dimC1d, rkd = residual_rank(Vd, Ed, G)
    assert eu == eu2 and ed == ed2, (eu, eu2, ed, ed2)
    # coker of sigma_r via explicit linear algebra on invariant cochains
    coker = coker_sigma(Vu, Eu, Vd, Ed, G)
    Q = cone_quotient(adj, m, K, r, G)
    return dict(eps_up=eu, eps_down=ed, D_r=eu-ed, coker=coker, Q=Q,
                ord_up=None, info_up=iu, info_down=idd)

def coker_sigma(Vu, Eu, Vd, Ed, G):
    """dim coker of R_G(up) -> R_G(down)."""
    vru, vlu = orbits_of(Vu, G, act); eru, elu = orbits_of(Eu, G, edge_action)
    vrd, vld = orbits_of(Vd, G, act); erd, eld = orbits_of(Ed, G, edge_action)
    def keeps(ereps):
        out = []
        for i, (e, orb) in enumerate(ereps):
            a, b = e
            if not any(act(g, a) == b and act(g, b) == a for g in G): out.append(i)
        return out
    ku, kd = keeps(eru), keeps(erd)
    kdi = {i: p for p, i in enumerate(kd)}
    # map each up non-reversible edge orbit to a down orbit index if the orbit lies downstairs
    img = np.zeros((len(kd), len(ku)))
    for col, i in enumerate(ku):
        e, orb = eru[i]
        a, b = e
        if a in vld and b in vld:            # whole orbit downstairs (G preserves j)
            j = eld[(a, b) if (a, b) in eld else (b, a)]
            if j in kdi: img[kdi[j], col] = 1.0
    Md = np.zeros((len(kd), len(vrd)))
    for i in kd:
        e, _ = erd[i]; a, b = e
        Md[kdi[i], vld[b]] += 1.0; Md[kdi[i], vld[a]] -= 1.0
    A = np.concatenate([img, Md], axis=1) if img.size or Md.size else np.zeros((0, 0))
    rA = 0 if A.size == 0 else int(np.linalg.matrix_rank(A, tol=TOL))
    rMd = 0 if Md.size == 0 else int(np.linalg.matrix_rank(Md, tol=TOL))
    eps_d = len(kd) - rMd
    return eps_d - (rA - rMd)

# ------------------------------------ ordinary K-token arena (plain subsets)
def ord_arena(adj, K):
    V = [frozenset(S) for S in itertools.combinations(sorted(adj), K)]
    Vset = set(V); E = set()
    for S in V:
        for u in S:
            for w in adj[u]:
                if w not in S:
                    t = frozenset(S - {u} | {w})
                    if t in Vset: E.add(frozenset({S, t}))
    return V, [tuple(sorted(e, key=lambda s: sorted(s))) for e in E]

def sact(g, S):  return frozenset(g[u] for u in S)
def sedge(g, e): 
    a, b = sact(g, e[0]), sact(g, e[1])
    return tuple(sorted((a, b), key=lambda s: sorted(s)))

def ord_residual(adj, K, G):
    """eps(X,K) = E_nr - V_orb + c  for the ordinary arena under group G."""
    V, E = ord_arena(adj, K)
    if not V: return 0, dict(E_nr=0, V_orb=0, c=0)
    vreps, vlab = orbits_of(V, G, sact)
    ereps, elab = orbits_of(E, G, sedge)
    nr = [e for (e, o) in ereps
          if not any(sact(g, e[0]) == e[1] and sact(g, e[1]) == e[0] for g in G)]
    par = list(range(len(vreps)))
    def find(x):
        while par[x] != x: par[x] = par[par[x]]; x = par[x]
        return x
    for (e, o) in ereps:
        ra, rb = find(vlab[e[0]]), find(vlab[e[1]])
        if ra != rb: par[ra] = rb
    c = len({find(i) for i in range(len(vreps))})
    return len(nr) - len(vreps) + c, dict(E_nr=len(nr), V_orb=len(vreps), c=c)

# ---- restricted group action on Y = X - m  = X - m -------------------------------
def deleted(adj, m, G):
    """Y = X-m relabelled 0..n-1, plus G restricted to Y, plus image of N(m)."""
    Y, idx, vs = induced_minus(adj, m)
    GY = []
    for g in G:
        GY.append(tuple(idx[g[v]] for v in vs))
    GY = list(dict.fromkeys(GY))          # dedupe (Stab(m) acts faithfully on Y if X connected)
    Nm = frozenset(idx[w] for w in adj[m])
    return Y, GY, Nm

def cross_orbits(Y, GY, Nm, k):
    pairs = [(frozenset(U), v) for U in itertools.combinations(sorted(Y), k)
             for v in sorted(Nm) if v not in U]
    reps, _ = orbits_of(pairs, GY, lambda g, x: (frozenset(g[u] for u in x[0]), g[x[1]]))
    return len(reps)

def split_predict(adj, m, K, r, G):
    """P_X(k) = eps(Y,k) + x_k - c_k ,  k = K-r-1."""
    k = K - r - 1
    Y, GY, Nm = deleted(adj, m, G)
    if k < 0 or k > len(Y): return None
    eps, info = ord_residual(Y, k, GY) if k > 0 else (0, dict(c=1))
    c = info['c'] if k > 0 else 1
    x = cross_orbits(Y, GY, Nm, k)
    return eps + x - c, dict(eps_Y=eps, x_k=x, c_k=c, k=k)

# --------- integer polynomial helpers (lists of coefficients, index = degree)
def pmul(A,B):
    C=[0]*(len(A)+len(B)-1)
    for i,a in enumerate(A):
        if a:
            for j,b in enumerate(B): C[i+j]+=a*b
    return C
def padd(A,B):
    n=max(len(A),len(B)); C=[0]*n
    for i,a in enumerate(A): C[i]+=a
    for i,b in enumerate(B): C[i]+=b
    return C
def pscale(A,s): return [a*s for a in A]
def pdivexact(A,B):
    """exact division A/B for integer polys; asserts zero remainder"""
    A=A[:]; q=[0]*(max(len(A)-len(B)+1,1))
    while len(A)>=len(B) and any(A):
        while A and A[-1]==0: A.pop()
        if len(A)<len(B): break
        d=len(A)-len(B); c=A[-1]//B[-1]
        assert c*B[-1]==A[-1], "inexact division"
        q[d]=c
        for i,b in enumerate(B): A[d+i]-=c*b
    while A and A[-1]==0: A.pop()
    assert not any(A), "nonzero remainder"
    return q
def coeff(A,k): return A[k] if 0<=k<len(A) else 0

def cycles_of(g, verts):
    seen=set(); out=[]
    for v in verts:
        if v in seen: continue
        c=[]; x=v
        while x not in seen:
            seen.add(x); c.append(x); x=g[x]
        out.append(c)
    return out

# --------------------- the closed form -----------------------------------
def closed_form(adj, m, G=None):
    """Returns (Enr(t), V(t), Xc(t), P1(t)) with P1 = Enr + Xc - V  (i.e. P_X(k)+c_k-1)."""
    if G is None: G = stab(adj, m)
    Y, GY, Nm = deleted(adj, m, G)
    verts = sorted(Y)
    Eg = [tuple(sorted(e)) for e in {frozenset((u,w)) for u in Y for w in Y[u]}]
    one=[1]; ONE_T=[1,1]; ONE_T2=[1,0,1]
    Enr=[0]; Vt=[0]; Xt=[0]
    for g in GY:
        cyc = cycles_of(g, verts)
        Z=[1]
        for c in cyc: 
            mono=[0]*(len(c)+1); mono[0]=1; mono[len(c)]=1
            Z=pmul(Z,mono)
        a = sum(1 for (u,w) in Eg if g[u]==u and g[w]==w)
        b = sum(1 for (u,w) in Eg if g[u]==w and g[w]==u)
        f = sum(1 for v in Nm if g[v]==v)
        Vt = padd(Vt, Z)
        if a: Enr = padd(Enr, pscale(pmul([0,1], pdivexact(Z, pmul(ONE_T,ONE_T))), a))
        if b: Enr = padd(Enr, pscale(pmul([0,1], pdivexact(Z, ONE_T2)), -b))
        if f: Xt  = padd(Xt , pscale(pdivexact(Z, ONE_T), f))
    n=len(GY)
    def dv(A):
        out=[]
        for a in A:
            assert a % n == 0, ("Burnside non-integer", a, n)
            out.append(a//n)
        return out
    Enr, Vt, Xt = dv(Enr), dv(Vt), dv(Xt)
    return Enr, Vt, Xt, padd(padd(Enr,Xt), pscale(Vt,-1))

def components_orbit_count(adj, m, G, k):
    """c_k = number of G-orbits of connected components of the k-token arena on X-m."""
    Y, GY, Nm = deleted(adj, m, G)
    comps = components(Y)
    sizes = [len(c) for c in comps]
    # component of A_k(Y) <-> distribution (k_1..k_s); G permutes the Y-components
    idx = {}
    for i,c in enumerate(comps):
        for v in c: idx[v]=i
    dists=[d for d in itertools.product(*[range(s+1) for s in sizes]) if sum(d)==k]
    def actd(g,d):
        out=[0]*len(comps)
        for i,c in enumerate(comps):
            j=idx[g[min(c)]]; out[j]=d[i]
        return tuple(out)
    reps,_=orbits_of(dists,GY,actd)
    return len(reps)

# ===== 2-cells, Hodge, level grading, exact rank =====


# ---------- 2-cells on the ordinary arena ----------
def arena_squares(adj, K, predicate="disjoint"):
    V,E = ord_arena(adj,K)
    Vset=set(V); eidx={}
    for i,e in enumerate(E): eidx[frozenset(e)]=i
    sqs=set()
    if predicate=="disjoint":
        for S in V:
            for u1 in S:
                for v1 in adj[u1]:
                    if v1 in S: continue
                    A=frozenset(S-{u1}|{v1})
                    for u2 in S:
                        if u2==u1 or u2==v1: continue
                        for v2 in adj[u2]:
                            if v2 in S or v2 in (u1,v1): continue
                            B=frozenset(S-{u2}|{v2})
                            Cc=frozenset(S-{u1,u2}|{v1,v2})
                            if A in Vset and B in Vset and Cc in Vset:
                                sqs.add(frozenset({S,A,B,Cc}))
    else:  # all arena 4-cycles
        nb={v:set() for v in V}
        for a,b in E: nb[a].add(b); nb[b].add(a)
        Vl=list(V)
        for i,a in enumerate(Vl):
            for b in nb[a]:
                for c in nb[b]:
                    if c==a: continue
                    for d in nb[c]&nb[a]:
                        if d==b: continue
                        sqs.add(frozenset({a,b,c,d}))
    out=[]
    for sq in sqs:
        q=list(sq)
        # order the 4 configs into a cycle
        nbl={x:[y for y in q if frozenset((x,y)) in eidx] for x in q}
        if any(len(nbl[x])!=2 for x in q): continue
        cyc=[q[0], nbl[q[0]][0]]
        while len(cyc)<4:
            nxt=[y for y in nbl[cyc[-1]] if y!=cyc[-2]]
            if not nxt: break
            cyc.append(nxt[0])
        if len(cyc)==4: out.append(cyc)
    return V,E,out,eidx

def harmonic(adj,K,G,predicate="disjoint"):
    V,E,SQ,eidx = arena_squares(adj,K,predicate)
    if not V: return None
    vreps,vlab = orbits_of(V,G,sact); ereps,elab = orbits_of(E,G,sedge)
    keep=[i for i,(e,o) in enumerate(ereps)
          if not any(sact(g,e[0])==e[1] and sact(g,e[1])==e[0] for g in G)]
    kidx={i:p for p,i in enumerate(keep)}
    # invariant 1-cochain -> value on an oriented edge
    def val(row, a, b):
        key=(a,b) if (a,b) in elab else (b,a)
        i=elab[key]
        if i not in kidx: return None,0
        sgn = 1 if ereps[i][0]==(a,b) else (-1 if ereps[i][0]==(b,a) else None)
        if sgn is None:
            # find the orientation of the representative relative to (a,b) via the group
            for g in G:
                if (sact(g,a),sact(g,b))==ereps[i][0]: sgn=1; break
                if (sact(g,b),sact(g,a))==ereps[i][0]: sgn=-1; break
        return kidx[i], sgn
    D0=np.zeros((len(keep),len(vreps)))
    for i in keep:
        a,b=ereps[i][0]; D0[kidx[i],vlab[b]]+=1; D0[kidx[i],vlab[a]]-=1
    D1=np.zeros((len(SQ),len(keep)))
    for r,cyc in enumerate(SQ):
        for t in range(4):
            a,b=cyc[t],cyc[(t+1)%4]
            j,s=val(None,a,b)
            if j is not None: D1[r,j]+=s
    r0 = 0 if D0.size==0 else int(np.linalg.matrix_rank(D0,tol=1e-9))
    r1 = 0 if D1.size==0 else int(np.linalg.matrix_rank(D1,tol=1e-9))
    eps = len(keep)-r0
    return dict(eps=eps, dimC1=len(keep), rank_d0=r0, rank_d1=r1,
                eps_harm=len(keep)-r1-r0, n_squares=len(SQ), V=len(V), E=len(E))


from itertools import combinations
def colsp(M,tol=1e-9):
    if M.size==0 or M.shape[1]==0: return np.zeros((M.shape[0],0))
    u,s,vt=np.linalg.svd(M,full_matrices=False); r=int((s>tol).sum()); return u[:,:r]
def nullsp(M,tol=1e-9):
    if M.shape[0]==0: return np.eye(M.shape[1])
    u,s,vt=np.linalg.svd(M); r=int((s>tol).sum()); return vt[r:].T
def dim_int(A,B,tol=1e-9):
    if A.shape[1]==0 or B.shape[1]==0: return 0
    return (np.linalg.matrix_rank(A,tol=tol)+np.linalg.matrix_rank(B,tol=tol)
            -np.linalg.matrix_rank(np.concatenate([A,B],axis=1),tol=tol))

def build(adj,K,G,predicate="disjoint"):
    V,E,SQ,_ = arena_squares(adj,K,predicate)
    Vi={c:i for i,c in enumerate(V)}
    key=lambda S: tuple(sorted(S))
    Eo=[]
    for (a,b) in E:
        A,B=(a,b) if key(a)<key(b) else (b,a)
        u=next(iter(A-B)); w=next(iter(B-A))
        Eo.append((A,B,u,w,frozenset(A&B)))
    m=len(Eo); Eidx={(A,B):i for i,(A,B,_,_,_) in enumerate(Eo)}
    def orient(a,b):
        if (a,b) in Eidx: return Eidx[(a,b)],1
        return Eidx[(b,a)],-1
    D0=np.zeros((m,len(V)))
    for i,(A,B,_,_,_) in enumerate(Eo): D0[i,Vi[B]]+=1; D0[i,Vi[A]]-=1
    D1=np.zeros((len(SQ),m))
    for r,cyc in enumerate(SQ):
        for t in range(4):
            j,s=orient(cyc[t],cyc[(t+1)%4]); D1[r,j]+=s
    # invariant 1-cochains from non-reversible oriented edge orbits
    done=set(); cols=[]
    for i,(A,B,_,_,_) in enumerate(Eo):
        if i in done: continue
        v=np.zeros(m); rev=False; touched=[]
        for g in G:
            j,s=orient(sact(g,A),sact(g,B))
            if v[j]!=0 and v[j]!=s: rev=True
            v[j]=s; touched.append(j)
        for j in touched: done.add(j)
        if not rev: cols.append(v)
    C1G=np.array(cols).T if cols else np.zeros((m,0))
    # invariant 0-cochains
    vr,vl = orbits_of(V,G,sact)
    V0=np.zeros((len(V),len(vr)))
    for c,i in vl.items(): V0[Vi[c],i]=1
    return Eo,V,SQ,D0,D1,C1G,V0

def levelF(adj,Eo,p):
    m=len(Eo); base={}
    for i,(A,B,u,w,S) in enumerate(Eo): base.setdefault(frozenset((u,w)),[]).append(i)
    cols=[]
    for be,idxs in base.items():
        u,w=sorted(be); rest=[v for v in adj if v not in be]
        for q in range(0,p+1):
            for T in combinations(rest,q):
                col=np.zeros(m); Tf=frozenset(T); nz=False
                for i in idxs:
                    A,B,uu,ww,S=Eo[i]
                    if Tf<=S: col[i]=1.0 if (uu,ww)==(u,w) else -1.0; nz=True
                if nz: cols.append(col)
    return np.array(cols).T if cols else np.zeros((m,0))

def analyse(adj,K,G,predicate="disjoint"):
    Eo,V,SQ,D0,D1,C1G,V0 = build(adj,K,G,predicate)
    t=C1G.shape[1]
    if t==0: return None
    Bi = colsp(D0@V0)
    eps = t - np.linalg.matrix_rank(Bi,tol=1e-9)
    M1 = D1@C1G if D1.shape[0] else np.zeros((0,t))
    Kd1 = C1G @ nullsp(M1)
    P = Kd1 - Bi@(Bi.T@Kd1) if Bi.shape[1] else Kd1
    H = colsp(P)
    prof=[]; prev=0
    for p in range(0,K):
        a=dim_int(H,levelF(adj,Eo,p)); prof.append(a-prev); prev=a
    return dict(eps=int(eps), eps_harm=H.shape[1], profile=prof,
                dimF0=np.linalg.matrix_rank(levelF(adj,Eo,0),tol=1e-9),
                invF0=dim_int(levelF(adj,Eo,0),C1G))


import itertools, random
from collections import Counter
def connected(a): return len(components(a))==1
def vorbs(g,A):
    seen=set(); out=[]
    for v in sorted(g):
        if v in seen: continue
        o=frozenset({h[v] for h in A}); seen|=o; out.append(o)
    return out

def c2_invariant(adj,K,G,predicate="disjoint"):
    """dim C^2_G : 2-cell orbits whose stabiliser contains no orientation-reversing element."""
    V,E,SQ,_=arena_squares(adj,K,predicate)
    def canon(cyc):
        n=len(cyc); best=None
        for s in range(n):
            for d in (1,-1):
                r=tuple(cyc[(s+d*i)%n] for i in range(n))
                key=tuple(tuple(sorted(x)) for x in r)
                if best is None or key<best[0]: best=(key,d)
        return best
    seen=set(); orbs=[]
    for cyc in SQ:
        k=canon(cyc)[0]
        if k in seen: continue
        orb=set(); rev=False
        for g in G:
            im=[sact(g,x) for x in cyc]
            kk,dd=canon(im)
            orb.add(kk)
            if kk==k:
                # does g map the square to itself with reversed orientation?
                base=[tuple(sorted(x)) for x in cyc]
                imk=[tuple(sorted(x)) for x in im]
                if imk!=base:
                    for s in range(4):
                        if [imk[(s+i)%4] for i in range(4)]==base: break
                        if [imk[(s-i)%4] for i in range(4)]==base: rev=True; break
        seen|=orb; orbs.append((cyc,rev))
    return sum(1 for _,r in orbs if not r), len(orbs)

def full(adj,K,G,predicate="disjoint"):
    Eo,V,SQ,D0,D1,C1G,V0 = build(adj,K,G,predicate)
    t=C1G.shape[1]
    if t==0: return None
    Bi=colsp(D0@V0); eps=t-np.linalg.matrix_rank(Bi,tol=1e-9)
    M1=D1@C1G if D1.shape[0] else np.zeros((0,t))
    r1=0 if M1.size==0 else int(np.linalg.matrix_rank(M1,tol=1e-9))
    c2nr,c2all=c2_invariant(adj,K,G,predicate)
    return dict(eps=int(eps), rank_d1=r1, eps_harm=int(eps)-r1, c2=c2nr, c2all=c2all)


from fractions import Fraction
P1,P2 = 2147483647, 2147483629

def rank_mod(M,p):
    A=[[int(x)%p for x in row] for row in M]; r=0; rows=len(A); cols=len(A[0]) if rows else 0
    for c in range(cols):
        piv=None
        for i in range(r,rows):
            if A[i][c]%p: piv=i; break
        if piv is None: continue
        A[r],A[piv]=A[piv],A[r]
        inv=pow(A[r][c],p-2,p)
        A[r]=[(x*inv)%p for x in A[r]]
        for i in range(rows):
            if i!=r and A[i][c]:
                f=A[i][c]; A[i]=[(A[i][j]-f*A[r][j])%p for j in range(cols)]
        r+=1
        if r==rows: break
    return r

def rank_exact(M):
    A=[[Fraction(int(x)) for x in row] for row in M]; r=0
    rows=len(A); cols=len(A[0]) if rows else 0
    for c in range(cols):
        piv=None
        for i in range(r,rows):
            if A[i][c]!=0: piv=i; break
        if piv is None: continue
        A[r],A[piv]=A[piv],A[r]
        pv=A[r][c]
        for i in range(rows):
            if i!=r and A[i][c]!=0:
                f=A[i][c]/pv
                A[i]=[A[i][j]-f*A[r][j] for j in range(cols)]
        r+=1
        if r==rows: break
    return r

def integer_model(adj,K,G,predicate="disjoint"):
    """M0 = d0 on invariant 0-cochains, M1 = d1, both in the orbit basis of C^1_G. Integer."""
    Eo,V,SQ,D0,D1,C1G,V0 = build(adj,K,G,predicate)
    t=C1G.shape[1]
    if t==0: return None
    # orbit representative coordinate: first index where the orbit vector is nonzero
    reps=[]
    for j in range(t):
        col=C1G[:,j]; i=int(np.flatnonzero(col)[0]); reps.append((i,int(round(col[i]))))
    R=np.zeros((t,C1G.shape[0]))
    for j,(i,s) in enumerate(reps): R[j,i]=s          # read-off map C^1_G -> orbit coords
    M0 = R @ (D0 @ V0)
    M1 = (D1 @ C1G)
    M0=np.rint(M0).astype(np.int64); M1=np.rint(M1).astype(np.int64)
    return t, M0.tolist(), M1.tolist()


def vorbs(g,A):
    seen=set(); out=[]
    for v in sorted(g):
        if v in seen: continue
        o=frozenset({h[v] for h in A}); seen|=o; out.append(o)
    return out
def edges_of(g): return [(a,b) for a in g for b in g[a] if a<b]
def girth(adj):
    best=10**9
    for s in adj:
        dist={s:0}; par={s:None}; q=deque([s])
        while q:
            x=q.popleft()
            for y in adj[x]:
                if y not in dist: dist[y]=dist[x]+1; par[y]=x; q.append(y)
                elif y!=par[x]: best=min(best,dist[x]+dist[y]+1)
    return best if best<10**9 else 0
def cone(g):
    n=len(g); return mkgraph(n+1, edges_of(g)+[(v,n) for v in g])
def subdivide(g):
    E=edges_of(g); n=len(g); out=[]
    for i,(a,b) in enumerate(E): out+=[(a,n+i),(n+i,b)]
    return mkgraph(n+len(E),out)
def multipartite(parts):
    v=0; bl=[]
    for p in parts: bl.append(list(range(v,v+p))); v+=p
    return mkgraph(v,[(a,b) for i,A in enumerate(bl) for j,B in enumerate(bl) if i<j for a in A for b in B])
def cyc_pend(n,sites):
    E=[(i,(i+1)%n) for i in range(n)]; nx=n
    for s in sites: E.append((s,nx)); nx+=1
    return mkgraph(nx,E)
BULL=mkgraph(5,[(0,1),(0,2),(0,3),(1,2),(1,4)])
CATERPILLAR=mkgraph(7,[(0,1),(1,2),(2,3),(1,4),(2,5),(3,6)])
CE8=mkgraph(8,[(0,1),(0,2),(0,3),(0,5),(0,7),(1,2),(1,4),(1,5),(2,3),(2,4),(2,6),
               (3,6),(3,7),(4,5),(4,6),(5,6),(5,7),(6,7)])
def site_reps(g):
    A=automorphisms(g); reps=[]; seen=set()
    for v in sorted(g):
        if v in seen: continue
        seen |= {h[v] for h in A}; reps.append(v)
    return reps


## 2. Reproduce the published zeros, and check the complex is real

Two things must hold before any nonzero result means anything: the published $\epsilon_{\mathrm{harm}}=0$
measurements must come back, and $d_1\circ d_0$ must be exactly zero — if the square orientations are
wrong, every number downstream is noise.

In [2]:
print("published: eps_harm = 0 across three arenas, both predicates")
for nm,g,K in [("C8",cycle(8),3),("C8",cycle(8),4),("Q3",qcube3(),3),("Petersen",petersen(),3)]:
    G=automorphisms(g)
    for pred in ("disjoint","all4"):
        h=harmonic(g,K,G,pred)
        print(f"  {nm:9s} K={K} |Aut|={len(G):3d} pred={pred:8s} cells={h['n_squares']:5d} "
              f"eps={h['eps']:3d} eps_harm={h['eps_harm']}")

print("\ncomplex validity:  d1 . d0 must be exactly 0")
worst=0.0; n=0; eul=0
for g,K in [(cycle(8),3),(qcube3(),3),(petersen(),3),(BULL,2),(BULL,3),(prism(),3),(path(6),3),(CE8,2)]:
    for pred in ("disjoint","all4"):
        G=automorphisms(g)
        Eo,V,SQ,D0,D1,C1G,V0 = build(g,K,G,pred)
        worst=max(worst,float(np.abs(D1@D0).max()) if D1.size else 0.0); n+=1
        c0,c1,c2=len(V),len(Eo),D1.shape[0]
        r0=int(np.linalg.matrix_rank(D0,tol=1e-9)); r1=0 if D1.size==0 else int(np.linalg.matrix_rank(D1,tol=1e-9))
        b0,b1,b2=c0-r0,c1-r0-r1,c2-r1
        eul += ((b0-b1+b2)==(c0-c1+c2))
print(f"  max |d1.d0| over {n} complexes = {worst:.3e}   -> valid: {worst==0.0}")
print(f"  Euler consistency  b0-b1+b2 == dimC0-dimC1+dimC2 : {eul}/{n}")

published: eps_harm = 0 across three arenas, both predicates
  C8        K=3 |Aut|= 16 pred=disjoint cells=   80 eps=  2 eps_harm=0
  C8        K=3 |Aut|= 16 pred=all4     cells=   80 eps=  2 eps_harm=0
  C8        K=4 |Aut|= 16 pred=disjoint cells=  120 eps=  3 eps_harm=0
  C8        K=4 |Aut|= 16 pred=all4     cells=  120 eps=  3 eps_harm=0
  Q3        K=3 |Aut|= 48 pred=disjoint cells=  168 eps=  1 eps_harm=0
  Q3        K=3 |Aut|= 48 pred=all4     cells=  306 eps=  1 eps_harm=0
  Petersen  K=3 |Aut|=120 pred=disjoint cells=  450 eps=  1 eps_harm=0
  Petersen  K=3 |Aut|=120 pred=all4     cells=  450 eps=  1 eps_harm=0

complex validity:  d1 . d0 must be exactly 0
  max |d1.d0| over 16 complexes = 0.000e+00   -> valid: True
  Euler consistency  b0-b1+b2 == dimC0-dimC1+dimC2 : 16/16


## 3. The witness, and what does *not* explain it

The bull is a triangle with a pendant on two of its three vertices. It is exhaustively minimal: no
connected graph on 4 vertices produces $\epsilon_{\mathrm{harm}}\neq0$, and every witness on 5
vertices has $|\mathrm{Aut}|=2$.

Three explanations die here, and the null results are the useful part:

* **Girth.** The §12 candidate class was high-girth. Petersen (girth 5), $C_8$+tail (8) and $C_9$ (9)
  all vanish; the witness has girth 3.
* **Winding.** Trees have $\beta_1(X)=0$ and still give $\epsilon_{\mathrm{harm}}>0$, so the surviving
  sector cannot be substrate winding. It is configuration-space structure — several tokens exchanging
  around essential vertices.
* **"It's a sliver."** Bridged $C_4$s reach $\epsilon_{\mathrm{harm}}=7$ at $K=2$ and $12$ at $K=3$.

In [3]:
print("exhaustive minimality over connected graphs on 4 and 5 vertices")
import itertools
for n in (4,5):
    allE=[(a,b) for a in range(n) for b in range(a+1,n)]; seen=set(); wit=set()
    for ne in range(n-1,len(allE)+1):
        for cE in itertools.combinations(allE,ne):
            g=mkgraph(n,cE)
            if len(components(g))!=1: continue
            k=min(tuple(sorted((min(p[a],p[b]),max(p[a],p[b])) for a in g for b in g[a] if a<b))
                  for p in itertools.permutations(range(n)))
            if k in seen: continue
            seen.add(k); G=automorphisms(g)
            for K in range(2,n):
                for pred in ("disjoint","all4"):
                    h=harmonic(g,K,G,pred)
                    if h and h['eps_harm']!=0: wit.add((k,len(G)))
    print(f"  n={n}: {len(seen)} iso classes, {len(wit)} witnesses"
          + (f", all with |Aut|={sorted({w[1] for w in wit})}" if wit else ""))

print("\ngirth is not the driver; trees are nonzero; the sector is not a sliver")
TWOC4=mkgraph(11,[(i,(i+1)%4) for i in range(4)]+[(4+i,4+((i+1)%4)) for i in range(4)]
                 +[(0,4),(1,8),(2,9),(5,10)])
rows=[("bull",BULL,[2,3]),("paw",paw(),[2,3]),("Petersen",petersen(),[3]),
      ("C9",cycle(9),[3]),("C8+tail",mkgraph(9,[(i,(i+1)%8) for i in range(8)]+[(0,8)]),[3]),
      ("caterpillar TREE",CATERPILLAR,[2,3]),("asym spider TREE",
        mkgraph(8,[(0,1),(1,2),(0,3),(3,4),(4,5),(0,6),(6,7)]),[2,3]),
      ("two C4s + pendants",TWOC4,[2,3])]
print(f"  {'substrate':20s} {'b1(X)':>5} {'girth':>5} {'|Aut|':>5} {'K':>2} {'eps':>4} {'e_harm':>6}")
for lab,g,Ks in rows:
    G=automorphisms(g); b1=len(edges_of(g))-len(g)+1; gi=girth(g)
    for K in Ks:
        h=harmonic(g,K,G,"disjoint")
        print(f"  {lab:20s} {b1:5d} {gi:5d} {len(G):5d} {K:2d} {h['eps']:4d} {h['eps_harm']:6d}")

exhaustive minimality over connected graphs on 4 and 5 vertices
  n=4: 6 iso classes, 0 witnesses
  n=5: 21 iso classes, 3 witnesses, all with |Aut|=[2]

girth is not the driver; trees are nonzero; the sector is not a sliver
  substrate            b1(X) girth |Aut|  K  eps e_harm
  bull                     1     3     2  2    2      1
  bull                     1     3     2  3    2      1
  paw                      1     3     2  2    0      0
  paw                      1     3     2  3    0      0
  Petersen                 6     5   120  3    1      0
  C9                       1     9    18  3    3      0
  C8+tail                  1     8     2  3   50      0
  caterpillar TREE         0     0     2  2    6      1
  caterpillar TREE         0     0     2  3   14      2
  asym spider TREE         0     0     2  2    7      0
  asym spider TREE         0     0     2  3   24      1
  two C4s + pendants       2     4     1  2   54      7
  two C4s + pendants       2     4     1  3  26

## 4. The counting obstruction

Exact forms lie in $\ker d_1$, so $d_1$ descends to the residual and

$$\epsilon_{\mathrm{harm}}=\epsilon-\operatorname{rank}\bigl(d_1|_{C^1_G}\bigr).$$

The rank of a linear map is bounded by the dimension of its codomain, and each invariant 2-cell
coordinate imposes at most one independent relation. Hence

$$\boxed{\;\epsilon_{\mathrm{harm}}\;\ge\;\epsilon-\dim C^2_G\;}\qquad\text{so}\qquad
\epsilon>\dim C^2_G\;\Longrightarrow\;\epsilon_{\mathrm{harm}}>0.$$

Rank–nullity, nothing more. It is vacuous on $C_8$, $Q_3$ and Petersen — there the invariant 2-cells
comfortably outnumber $\epsilon$ — which is exactly why it was never written down.

**It resolves the paw/bull pair mechanically.** The paw has no invariant non-exact direction at all.
The bull has two, and exactly **one** invariant 2-cell: one relation cannot close two channels, so one
must survive. $W_6$ has three channels and three relations, enough to close them, and does.

$\dim C^2_G$ counts 2-cell orbits whose stabiliser contains no orientation-reversing element — the same
reversibility rule that defines $\dim C^1_G$ in Theorem 2.1.

In [4]:
print(f"{'graph':13s} {'K':>2} {'eps':>4} {'rk d1':>6} {'e_harm':>7} {'dimC2_G':>8} {'bound':>6} {'verdict':>8}")
for nm,g,K in [("C8",cycle(8),3),("Q3",qcube3(),3),("Petersen",petersen(),3),
               ("paw",paw(),2),("bull",BULL,2),("bull",BULL,3),
               ("wheel6",mkgraph(6,[(0,i) for i in range(1,6)]+[(i,i%5+1) for i in range(1,6)]),3),
               ("caterpillar",CATERPILLAR,2)]:
    G=automorphisms(g); r=full(g,K,G); b=max(0,r['eps']-r['c2'])
    v = "TIGHT" if (b>0 and b==r['eps_harm']) else ("forces>0" if b>0 else "silent")
    print(f"{nm:13s} {K:2d} {r['eps']:4d} {r['rank_d1']:6d} {r['eps_harm']:7d} {r['c2']:8d} {b:6d} {v:>8s}")

graph          K  eps  rk d1  e_harm  dimC2_G  bound  verdict
C8             3    2      2       0        4      0   silent
Q3             3    1      1       0        3      0   silent
Petersen       3    1      1       0        2      0   silent
paw            2    0      0       0        0      0   silent
bull           2    2      1       1        1      1    TIGHT
bull           3    2      1       1        1      1    TIGHT
wheel6         3    3      3       0        3      0   silent
caterpillar    2    6      5       1        5      1    TIGHT


In [5]:
import itertools
from collections import Counter
allE=[(a,b) for a in range(6) for b in range(a+1,6)]
sn=set(); reps6=[]
for ne in range(5,16):
    for cE in itertools.combinations(allE,ne):
        g=mkgraph(6,cE)
        if len(components(g))!=1: continue
        k=min(tuple(sorted((min(p[a],p[b]),max(p[a],p[b])) for a in g for b in g[a] if a<b))
              for p in itertools.permutations(range(6)))
        if k in sn: continue
        sn.add(k); reps6.append(g)
print(f"exhaustive: {len(reps6)} connected graphs on 6 vertices, K=2,3, disjoint-square predicate\n")
n=viol=active=act_hit=tight=nz=cert=0; orb=Counter()
for g in reps6:
    A=automorphisms(g); vo=len(vorbs(g,A))
    for K in (2,3):
        r=full(g,K,A)
        if r is None: continue
        n+=1; b=max(0,r['eps']-r['c2'])
        viol += (r['eps_harm']<b)
        if b>0:
            active+=1; act_hit += (r['eps_harm']>0); tight += (b==r['eps_harm'])
        if r['eps_harm']>0: nz+=1; cert += (b>0)
        orb[(vo, r['eps_harm']>0)]+=1
print(f"  cases {n}   bound violations: {viol}")
print(f"  bound active: {active};  of those eps_harm>0: {act_hit}   ({'no false positives' if active==act_hit else 'FALSE POSITIVE'})")
print(f"  bound exactly tight when active: {tight}/{active}")
print(f"  coverage: certifies {cert}/{nz} of all nonzero cases ({cert/nz*100:.0f}%)")
print(f"\n  hit rate by vertex-orbit count (the better predictor; |Aut| is not monotone):")
for vo in sorted({k[0] for k in orb}):
    h=orb[(vo,True)]; z=orb[(vo,False)]
    print(f"    {vo} orbits: {h:3d}/{h+z:3d}")

exhaustive: 112 connected graphs on 6 vertices, K=2,3, disjoint-square predicate

  cases 222   bound violations: 0
  bound active: 85;  of those eps_harm>0: 85   (no false positives)
  bound exactly tight when active: 61/85
  coverage: certifies 85/98 of all nonzero cases (87%)

  hit rate by vertex-orbit count (the better predictor; |Aut| is not monotone):
    1 orbits:   0/  8
    2 orbits:   0/ 36
    3 orbits:  17/ 54
    4 orbits:  31/ 66
    5 orbits:  34/ 42
    6 orbits:  16/ 16


## 5. The vanishing side, and why the predicate decides it

The obstruction only ever proves $\epsilon_{\mathrm{harm}}>0$. Vanishing is the opposite estimate —
$d_1$ must have *full* rank on the residual — and the counting bound is structurally silent there.

Stated as a chain-level question:

$$\epsilon_{\mathrm{harm}}=0\iff d_1:C^1_G/B^1_G\to C^2_G\ \text{is injective}.$$

Empirically $\#(V/\mathrm{Aut})\le2$ looked sufficient. It is not, and the failure is instructive:
the statement is **true for one 2-cell predicate and false for the other**.

In [6]:
print("minimal counterexample, disjoint-support-square predicate")
A=automorphisms(CE8); O=vorbs(CE8,A)
print(f"  edges = {sorted(edges_of(CE8))}")
print(f"  |V|=8  |Aut|={len(A)}  vertex orbits = {[sorted(o) for o in O]}  degseq={sorted(len(CE8[v]) for v in CE8)}")
print(f"\n  EXACT RATIONAL rank (no floating point):")
for K in (2,3):
    for pred in ("disjoint","all4"):
        t,M0,M1 = integer_model(CE8,K,A,pred)
        r0=rank_exact(M0); r1=rank_exact(M1) if M1 and M1[0] else 0
        print(f"    K={K} {pred:8s} t={t:3d} rank_Q(M0)={r0:3d} rank_Q(M1)={r1:3d} "
              f"sum={r0+r1}/{t}   eps={t-r0:3d}  eps_harm={t-r0-r1}")
print("\n  Same graph, same group, same 1-skeleton: eps is identical, eps_harm is 1 or 0")
print("  according to which 2-cells are declared. That is the predicate-dependence, at the")
print("  level of the conjecture's truth value.")

minimal counterexample, disjoint-support-square predicate
  edges = [(0, 1), (0, 2), (0, 3), (0, 5), (0, 7), (1, 2), (1, 4), (1, 5), (2, 3), (2, 4), (2, 6), (3, 6), (3, 7), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)]
  |V|=8  |Aut|=8  vertex orbits = [[0, 2, 5, 6], [1, 3, 4, 7]]  degseq=[4, 4, 4, 4, 5, 5, 5, 5]

  EXACT RATIONAL rank (no floating point):
    K=2 disjoint t= 14 rank_Q(M0)=  6 rank_Q(M1)=  7 sum=13/14   eps=  8  eps_harm=1
    K=2 all4     t= 14 rank_Q(M0)=  6 rank_Q(M1)=  8 sum=14/14   eps=  8  eps_harm=0
    K=3 disjoint t= 32 rank_Q(M0)=  9 rank_Q(M1)= 22 sum=31/32   eps= 23  eps_harm=1
    K=3 all4     t= 32 rank_Q(M0)=  9 rank_Q(M1)= 23 sum=32/32   eps= 23  eps_harm=0

  Same graph, same group, same 1-skeleton: eps is identical, eps_harm is 1 or 0
  according to which 2-cells are declared. That is the predicate-dependence, at the
  level of the conjecture's truth value.


### Why a modular rank is a proof here

$d_1d_0=0$ forces $\operatorname{rank}_{\mathbb Q}M_0+\operatorname{rank}_{\mathbb Q}M_1\le t$, and
$\operatorname{rank}_p\le\operatorname{rank}_{\mathbb Q}$ always. So if the mod-$p$ ranks already sum
to $t$, the rational ranks are squeezed to equality and $\epsilon_{\mathrm{harm}}=0$ is **proved** for
that case, not observed. Two primes are used, and the matrices are integer throughout — the earlier
programme's tolerance failure (a relative tolerance reporting full rank on entries of size $10^{-16}$)
cannot occur here.

In [ ]:
import random
pool=[]
for g in reps6:
    A=automorphisms(g)
    if len(vorbs(g,A))<=2: pool.append(("6v",g))
for nm,g in [("C8",cycle(8)),("C9",cycle(9)),("C10",cycle(10)),("Q3",qcube3()),("Petersen",petersen()),
             ("prism",prism()),("K6",complete(6)),("K7",complete(7)),("star7",star(7)),("star8",star(8)),
             ("wheel6",mkgraph(6,[(0,i) for i in range(1,6)]+[(i,i%5+1) for i in range(1,6)])),
             ("wheel7",mkgraph(7,[(0,i) for i in range(1,7)]+[(i,i%6+1) for i in range(1,7)])),
             ("K_{2,4}",multipartite([2,4])),("K_{2,5}",multipartite([2,5])),("K_{3,5}",multipartite([3,5])),
             ("K_{1,3,3}",multipartite([1,3,3])),("K_{2,2,3}",multipartite([2,2,3])),
             ("octahedron",mkgraph(6,[(a,b) for a in range(6) for b in range(a+1,6) if b-a!=3])),
             ("pent.prism",mkgraph(10,[(i,(i+1)%5) for i in range(5)]+[(5+i,5+(i+1)%5) for i in range(5)]+[(i,5+i) for i in range(5)])),
             ("cone(C5)",cone(cycle(5))),("cone(C6)",cone(cycle(6))),("cone(K4)",cone(complete(4))),
             ("subdiv(K4)",subdivide(complete(4))),("subdiv(C5)",subdivide(cycle(5))),
             ("subdiv(C6)",subdivide(cycle(6)))]:
    if len(vorbs(g,automorphisms(g)))<=2: pool.append((nm,g))
random.seed(23)
for n in (7,8,9):
    aE=[(a,b) for a in range(n) for b in range(a+1,n)]; got=0
    for _ in range(40000):
        if got>=6: break
        g=mkgraph(n,random.sample(aE,random.randint(n-1,min(3*n,len(aE)))))
        if len(components(g))!=1: continue
        A=automorphisms(g)
        if len(vorbs(g,A))!=2: continue
        pool.append((f"rand{n}_{got}",g)); got+=1
print(f"pool: {len(pool)} substrates with at most 2 vertex orbits")
print("(deliberately adversarial: cones over vertex-transitive graphs carry large eps;")
print(" subdivisions are 2-orbit AND high-girth, the class the old conjecture pointed at)\n")
res={"disjoint":[0,0,[]],"all4":[0,0,[]]}
for nm,g in pool:
    A=automorphisms(g)
    if len(g)>22: continue
    for K in (2,3):
        if len(g)>14 and K>2: continue
        for pred in ("disjoint","all4"):
            im=integer_model(g,K,A,pred)
            if im is None: continue
            t,M0,M1=im
            r0=rank_mod(M0,P1); r1=rank_mod(M1,P1) if M1 and M1[0] else 0
            r0b=rank_mod(M0,P2); r1b=rank_mod(M1,P2) if M1 and M1[0] else 0
            res[pred][0]+=1
            if r0+r1==t and r0b+r1b==t: res[pred][1]+=1
            else: res[pred][2].append((nm,len(g),K,t-r0,t-r0-r1))
for pred in ("disjoint","all4"):
    n,ok,bad=res[pred]
    print(f"  predicate {pred:8s}: {ok}/{n} PROVED eps_harm=0    counterexamples: {len(bad)}")
    for b in bad: print(f"       {b[0]:10s} |V|={b[1]} K={b[2]} eps={b[3]} eps_harm={b[4]}")

pool: 60 substrates with at most 2 vertex orbits
(deliberately adversarial: cones over vertex-transitive graphs carry large eps;
 subdivisions are 2-orbit AND high-girth, the class the old conjecture pointed at)



## 6. The level grading, anchored

The spectator (Johnson) filtration $F_p$ grades an arena cochain by how many spectator tokens it
depends on: via Lemma 1.1 an arena edge is a pair (base edge $e$, spectator set $S$), and $F_p$ is
spanned by $\mathbb 1[T\subseteq S]$ with $|T|\le p$.

Level 0 is not a new object — it **is** the reader space $\pi^*C^1(X)$ of Theorem 3.1. That gives an
independent anchor: the published §9 values of $\dim\pi^*C^1(X)$ and its invariant part must come back
exactly, including $P_8$'s nontrivial invariant part of 3.

In [ ]:
print("anchor: level 0 == pi*C^1(X), against published Section 9 values")
for nm,g,K,dexp,iexp in [("C8",cycle(8),3,8,0),("C8",cycle(8),4,8,0),("Q3",qcube3(),3,12,0),
                         ("Petersen",petersen(),3,15,0),("P8",path(8),3,7,3)]:
    G=automorphisms(g); r=analyse(g,K,G)
    ok=(r['dimF0']==dexp and r['invF0']==iexp)
    print(f"  {nm:9s} K={K}  dim F_0={r['dimF0']:3d} (paper {dexp})  invariant part={r['invF0']} "
          f"(paper {iexp})   {'OK' if ok else 'MISMATCH'}")

print("\nharmonic profile by level  [p=0, p=1, p=2, ...]")
print(f"  {'substrate':24s} {'b1(X)':>5} {'K':>2} {'eps':>4} {'e_harm':>6}  profile")
for lab,g,Ks in [("bull",BULL,[2,3]),("C5 + 2 pendants",cyc_pend(5,[0,1]),[2,3]),
                 ("caterpillar  TREE",CATERPILLAR,[2,3]),
                 ("asym spider  TREE",mkgraph(8,[(0,1),(1,2),(0,3),(3,4),(4,5),(0,6),(6,7)]),[2,3]),
                 ("C8",cycle(8),[3]),("Petersen",petersen(),[3])]:
    G=automorphisms(g); b1=len(edges_of(g))-len(g)+1
    for K in Ks:
        r=analyse(g,K,G,"disjoint")
        print(f"  {lab:24s} {b1:5d} {K:2d} {r['eps']:4d} {r['eps_harm']:6d}  {[int(x) for x in r['profile']]}")
print("\n  Reported as measurement only. An earlier reading of these profiles as a symmetry law")
print("  was tested exhaustively and killed -- see the ledger below.")

## 7. Audit table

One row per $(X,K,\text{predicate})$, carrying everything a later reader needs to re-derive the
claims: orbit counts, $\epsilon$, $\operatorname{rank}d_1$, $\epsilon_{\mathrm{harm}}$, and whether the
counting bound is active, tight or silent.

In [ ]:
rows=[]
cases=[("C8",cycle(8),3),("Q3",qcube3(),3),("Petersen",petersen(),3),("prism",prism(),3),
       ("paw",paw(),2),("bull",BULL,2),("bull",BULL,3),
       ("wheel6",mkgraph(6,[(0,i) for i in range(1,6)]+[(i,i%5+1) for i in range(1,6)]),3),
       ("caterpillar",CATERPILLAR,2),("caterpillar",CATERPILLAR,3),
       ("C5+2pend",cyc_pend(5,[0,1]),2),("CE8",CE8,2),("CE8",CE8,3)]
hdr=f"{'substrate':12s} {'K':>2} {'pred':>8s} {'|Aut|':>5} {'Vorb':>4} {'Eorb':>4} {'girth':>5} " \
    f"{'eps':>4} {'rkd1':>5} {'C2_G':>5} {'eharm':>5} {'bound':>6}"
print(hdr); print("-"*len(hdr))
for nm,g,K in cases:
    A=automorphisms(g)
    eo=len({frozenset({tuple(sorted((h[a],h[b]))) for h in A}) for a,b in edges_of(g)})
    for pred in ("disjoint","all4"):
        r=full(g,K,A,pred)
        if r is None: continue
        b=max(0,r['eps']-r['c2'])
        st="tight" if (b>0 and b==r['eps_harm']) else ("active" if b>0 else "silent")
        print(f"{nm:12s} {K:2d} {pred:>8s} {len(A):5d} {len(vorbs(g,A)):4d} {eo:4d} {girth(g):5d} "
              f"{r['eps']:4d} {r['rank_d1']:5d} {r['c2']:5d} {r['eps_harm']:5d} {st:>6s}")

## 8. Ledger

### Established

| | |
|---|---|
| $\epsilon_{\mathrm{harm}}\neq0$ occurs | §12 item 2 answered in the affirmative |
| minimal witness | bull, $\lvert V\rvert=5$, $\lvert\mathrm{Aut}\rvert=2$; exhaustively minimal on $\le5$ vertices |
| girth is not the driver | Petersen 5, $C_8$+tail 8, $C_9$ 9 all vanish; witness at girth 3 |
| the sector is not winding | trees, $\beta_1(X)=0$, give $\epsilon_{\mathrm{harm}}>0$ |
| counting obstruction | $\epsilon_{\mathrm{harm}}\ge\epsilon-\dim C^2_G$; no violations, no false positives, 87% coverage |
| paw/bull resolved | two channels, one relation |
| $\epsilon$ vs $\epsilon_{\mathrm{harm}}$ | $\epsilon$ is 1-skeleton; $\epsilon_{\mathrm{harm}}$ is 2-skeleton and predicate-dependent |
| level 0 anchored | equals $\pi^*C^1(X)$, matching published §9 values including $P_8$'s 3 |

### Conjectural

$\#(V/\mathrm{Aut})\le2\Rightarrow\epsilon_{\mathrm{harm}}=0$, **all-4-cycle predicate only**, proved
case-by-case on the tested pool. **False** for the disjoint-square predicate — minimal counterexample
$\lvert V\rvert=8$, two orbits, $\lvert\mathrm{Aut}\rvert=8$.

### Killed during this work — both mine, neither reached a claim

1. *"$\epsilon_{\mathrm{harm}}$ hit rate is monotone in $\lvert\mathrm{Aut}\rvert$."* Overstated. The
   reported $8\to12$ reversal is $1/18$ against $2/20$ — noise in 18- and 20-case bins. $\lvert\mathrm{Aut}\rvert$
   is not the right variable; vertex-orbit count is monotone in the same data where $\lvert\mathrm{Aut}\rvert$
   is not. **Recorded as a candidate control parameter, not an order parameter.**
2. *"With nontrivial $\mathrm{Aut}$, harmonic classes live at level $p\ge1$."* Proposed from the bull,
   the pendant cycles and the trees, which all profile $[0,1,\dots]$. Exhaustive 6-vertex: **72 of 82**
   symmetric cases carry level-0 content. The law held only on the hand-picked family — cataloguing
   instances, not testing.

The second one is why §6's profiles are reported as measurements and nothing is built on them.

### Boundary

No physical identification is made. $\epsilon_{\mathrm{harm}}$ is the dimension of an invariant
harmonic cohomology sector under a declared 2-cell predicate. Calling it a force, a curvature channel
or a price requires an independently defined map that is not constructed here.

The revision this forces on *Transformation without loss* §2 is narrow and exact: *the invariant
non-exact sector is coexact in a restricted high-symmetry regime, not in general.* $C_8$, $Q_3$ and
Petersen are vertex-transitive and sit inside that regime; they are not generic evidence.

### Open

1. Prove or refute $\#(V/\mathrm{Aut})\le2\Rightarrow\epsilon_{\mathrm{harm}}=0$ for the all-4-cycle
   complex. The target is injectivity of $d_1$ on $C^1_G/B^1_G$, and the two possible quotient
   incidence patterns at $\le2$ orbits are few enough to attempt directly.
2. The 13 nonzero cases the counting bound misses. Those turn on dependence *among* the 2-cell
   relations, which counting cannot see.
3. Characterise which pairs of predicates can disagree. The $\lvert V\rvert=8$ counterexample shows
   disjoint-squares and all-4-cycles are genuinely different theories on the same 1-skeleton; what
   separates them is unknown.